In [2]:
import pandas as pd
import numpy as np
import datetime as dt

# GIAI ĐOẠN 1: LÀM SẠCH DỮ LIỆU 

In [3]:
file_path = '../data/raw/superstore.csv' 
try:
    # Nếu chạy lỗi file path
    df = pd.read_csv(file_path)
    print(f"Đã tải dữ liệu thành công. Kích thước: {df.shape}")
except FileNotFoundError:
    print("Không tìm thấy file. Vui lòng kiểm tra lại đường dẫn.")
    # test nhanh
    df = pd.read_csv('superstore.csv')


Đã tải dữ liệu thành công. Kích thước: (51290, 27)


In [5]:
# Chuẩn hóa tên cột
df.columns = df.columns.str.replace('.', '_').str.lower()

# Xóa các cột rác không có giá trị phân tích
junk_cols = ['row_id', '记录数', 'weeknum', 'market2']
df.drop(columns=[c for c in junk_cols if c in df.columns], inplace=True, errors='ignore')

# Chuyển đổi định dạng thời gian
df['order_date'] = pd.to_datetime(df['order_date'])
df['ship_date'] = pd.to_datetime(df['ship_date'])

# Lọc bỏ các ngày logic sai (Ship Date < Order Date)
df = df[df['ship_date'] >= df['order_date']]

# Xử lý trùng lặp nghiệp vụ (Order ID + Product ID) - Giữ lại dòng cập nhật mới nhất
df = df.sort_values('order_date').drop_duplicates(subset=['order_id', 'product_id'], keep='last')

# GIAI ĐOẠN 2: FEATURE ENGINEERING

In [6]:
# Dữ liệu Thời gian
df['order_year'] = df['order_date'].dt.year
df['order_month'] = df['order_date'].dt.month
df['order_quarter'] = "Q" + df['order_date'].dt.quarter.astype(str)

# Dữ liệu Vận hành
df['delivery_days'] = (df['ship_date'] - df['order_date']).dt.days
df['is_late_delivery'] = df['delivery_days'].apply(lambda x: 1 if x > 5 else 0)

# Dữ liệu Tài chính
df['profit_margin'] = np.where(df['sales'] > 0, df['profit'] / df['sales'], 0)
df['is_loss_maker'] = df['profit'].apply(lambda x: 1 if x < 0 else 0)

def categorize_discount(x):
    if x == 0: return 'No Discount'
    elif x < 0.2: return 'Low (<20%)'
    elif x <= 0.5: return 'Medium (20-50%)'
    else: return 'High (>50%)'
df['discount_level'] = df['discount'].apply(categorize_discount)

# GIAI ĐOẠN 3: TÍNH TOÁN MÔ HÌNH RFM

In [8]:
# Chọn ngày chốt báo cáo (Snapshot Date) là ngày lớn nhất trong tập dữ liệu + 1 ngày
snapshot_date = df['order_date'].max() + dt.timedelta(days=1)

# Tính R, F, M cho từng khách hàng
rfm = df.groupby('customer_id').agg({
    'order_date': lambda x: (snapshot_date - x.max()).days, # Recency
    'order_id': 'nunique',                                  # Frequency
    'sales': 'sum'                                          # Monetary
}).reset_index()

rfm.rename(columns={'order_date': 'recency', 'order_id': 'frequency', 'sales': 'monetary'}, inplace=True)

# Chấm điểm từ 1-5 dùng hàm qcut (rank method='first' để tránh lỗi trùng lặp phân vị)
r_labels = range(5, 0, -1) # Recency càng nhỏ càng tốt (5 là tốt nhất)
f_labels = range(1, 6)     # Frequency càng lớn càng tốt
m_labels = range(1, 6)     # Monetary càng lớn càng tốt

rfm['r_score'] = pd.qcut(rfm['recency'].rank(method='first'), q=5, labels=r_labels)
rfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=5, labels=f_labels)
rfm['m_score'] = pd.qcut(rfm['monetary'].rank(method='first'), q=5, labels=m_labels)

# Gộp điểm thành chuỗi (vd: '555')
rfm['rfm_score'] = rfm['r_score'].astype(str) + rfm['f_score'].astype(str) + rfm['m_score'].astype(str)

# Hàm Gán nhãn Phân khúc khách hàng (Segmentation)
def segment_customer(row):
    r, f, m = int(row['r_score']), int(row['f_score']), int(row['m_score'])
    if r >= 4 and f >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3:
        return 'Loyal Customers'
    elif r <= 2 and f >= 3:
        return 'At Risk'
    elif r <= 2 and f <= 2:
        return 'Lost Customers'
    else:
        return 'Potential/Others'

rfm['customer_segment'] = rfm.apply(segment_customer, axis=1)

# Nối dữ liệu RFM trở lại bảng chính
df_final = df.merge(rfm[['customer_id', 'recency', 'frequency', 'monetary', 'rfm_score', 'customer_segment']], on='customer_id', how='left')

# GIAI ĐOẠN 4: XUẤT DỮ LIỆU FINAL 

In [ ]:
final_file_path = '../data/processed/superstore_final.csv'
df_final.to_csv(final_file_path, index=False)
print(f"Lưu thành công ({df_final.shape[0]} dòng, {df_final.shape[1]} cột) đã lưu tại: {final_file_path}")

Lưu thành công (51252 dòng, 36 cột) đã lưu tại: ../data/processed/superstore_final.csv


: 